# 07 K-means : regrouper des soudures similaires sans la cible

Nous cherchons des groupes de soudures proches par leur **composition et leurs paramètres de soudage**, sans utiliser UTS, l'allongement ni le score. Le score n'intervient qu'après coup, pour décrire les groupes obtenus.

Comme la cible n'est pas nécessaire, nous pouvons utiliser **toutes les lignes hors test**, y compris celles sans mesure de traction. Les groupes du test restent écartés.

*Texte d'introduction à rédiger.*

## 1 Charger les données hors test

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn import __version__ as sklearn_version
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             adjusted_rand_score, mean_squared_error)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED = ROOT / 'data/processed'
OUT = ROOT / 'outputs/07'
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42

fichiers = ['welddb_clean.csv', 'welddb_groups.csv', 'welddb_metadata.json']
if not all((PROCESSED / nom).exists() for nom in fichiers):
    import subprocess, sys
    subprocess.run([sys.executable, str(ROOT / 'src/export_processed.py')], check=True)

In [ ]:
categories = ['AC_DC', 'Electrode_pol', 'WeldType']
propre = pd.read_csv(PROCESSED / 'welddb_clean.csv', index_col='row_id',
    dtype={c: str for c in categories + ['WeldID', 'hardness_scale']},
    keep_default_na=False, na_values=[''], float_precision='round_trip')
groupes = pd.read_csv(PROCESSED / 'welddb_groups.csv', index_col='row_id')['groupe']
schema = json.loads((PROCESSED / 'welddb_metadata.json').read_text())['schema']
variables_X = schema['COMPOSITION'] + schema['PROCESS']

separation = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
idx_dev, idx_test = next(separation.split(propre, groups=groupes))
hors_test = propre.index[idx_dev]

traction = pd.read_csv(ROOT / 'outputs/02/dataset_traction.csv', index_col='row_id',
                       dtype={'partition': str}, keep_default_na=False, na_values=[''])
assert set(traction.index[traction['partition'].eq('developpement')]) <= set(hors_test)
assert set(traction.index[traction['partition'].eq('test')]).isdisjoint(hors_test)

X = propre.loc[hors_test, variables_X]
G = groupes.loc[hors_test]
etiquetees = X.index.intersection(traction.index)
print(f'{len(X)} lignes hors test, dont {len(etiquetees)} avec un score et '
      f'{len(X) - len(etiquetees)} sans mesure de traction.')

## 2 Préparer les entrées

K-means repose sur des **distances euclidiennes** : les variables doivent être standardisées. Nous gardons les variables numériques remplies à au moins 25 % (même seuil que le notebook 03), imputées par la médiane. Comme pour l'ACP du notebook 04, les variables catégorielles ne participent pas au calcul ; elles servent à décrire les groupes ensuite.

Limite : une variable très souvent imputée crée beaucoup de valeurs identiques, ce qui peut produire des groupes liés aux **valeurs manquantes** plutôt qu'à la métallurgie. Nous le vérifierons à la section 5.

In [ ]:
taux = X.notna().mean()
numeriques = [c for c in taux[taux >= 0.25].index if c not in categories]
print('Variables écartées :', [c for c in variables_X if c not in numeriques and c not in categories])

preparation = Pipeline([('mediane', SimpleImputer(strategy='median')),
                        ('standardisation', StandardScaler())])
Z = pd.DataFrame(preparation.fit_transform(X[numeriques]), index=X.index, columns=numeriques)
print(f'K-means sur {Z.shape[0]} lignes et {Z.shape[1]} variables numériques standardisées.')

## 3 Choisir le nombre de groupes

Pour k de 2 à 12, nous calculons :
- l'**inertie** (somme des distances au carré au centre), qui baisse toujours quand k augmente : on cherche un coude ;
- le **score de silhouette** (entre -1 et 1, plus haut = groupes mieux séparés) ;
- l'indice de **Davies-Bouldin** (plus bas = mieux) ;
- la **stabilité** : nous refaisons K-means sur 8 tirages avec remise des lignes (bootstrap), puis comparons la partition obtenue sur toutes les lignes à la partition de référence (indice de Rand ajusté, 1 = partitions identiques). Des groupes qui changent beaucoup d'un tirage à l'autre ne sont pas fiables.

In [ ]:
valeurs_k = range(2, 13)
tirage = np.random.default_rng(SEED)
criteres = []
for k in valeurs_k:
    modele = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(Z)
    stabilite = np.mean([
        adjusted_rand_score(modele.labels_, KMeans(n_clusters=k, n_init=10, random_state=s)
                            .fit(Z.iloc[tirage.choice(len(Z), len(Z))]).predict(Z))
        for s in range(8)])
    criteres.append({'k': k, 'inertie': modele.inertia_,
                     'silhouette': silhouette_score(Z, modele.labels_),
                     'davies_bouldin': davies_bouldin_score(Z, modele.labels_),
                     'stabilite_ARI': stabilite,
                     'plus_petit_groupe': np.bincount(modele.labels_).min()})
criteres = pd.DataFrame(criteres).set_index('k')
display(criteres.round(3))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, (colonne, titre) in zip(axes, [('inertie', 'Inertie (coude)'),
                                       ('silhouette', 'Silhouette (plus haut = mieux)'),
                                       ('davies_bouldin', 'Davies-Bouldin (plus bas = mieux)'),
                                       ('stabilite_ARI', 'Stabilité entre initialisations')]):
    ax.plot(criteres.index, criteres[colonne], marker='o')
    ax.set(xlabel='k', title=titre)
fig.tight_layout(); fig.savefig(OUT / 'choix_k.png', dpi=150); plt.show()

Règle retenue : parmi les k dont la stabilité dépasse 0,75, nous prenons celui qui a la meilleure silhouette, en excluant k = 2 qui sépare souvent seulement quelques lignes extrêmes du reste.

*Choix à discuter et justifier dans le rapport ; la variable `K` peut être modifiée à la main.*

In [ ]:
candidats = criteres[(criteres.index > 2) & (criteres['stabilite_ARI'] >= 0.75)]
K = int((candidats if len(candidats) else criteres.loc[3:])['silhouette'].idxmax())
kmeans = KMeans(n_clusters=K, n_init=20, random_state=SEED).fit(Z)
cluster = pd.Series(kmeans.labels_, index=Z.index, name='cluster')
print(f'k retenu : {K}')
display(cluster.value_counts().sort_index().to_frame('lignes'))

## 4 Visualiser les groupes sur les deux premiers axes de l'ACP

L'ACP est calculée sur les mêmes données standardisées. Elle sert uniquement à dessiner : K-means a travaillé dans toutes les dimensions, donc des groupes qui se chevauchent sur ce plan peuvent être séparés sur d'autres axes.

In [ ]:
acp = PCA(n_components=2, random_state=SEED).fit(Z)
plan = acp.transform(Z)
centres_plan = acp.transform(pd.DataFrame(kmeans.cluster_centers_, columns=numeriques))
fig, ax = plt.subplots(figsize=(8, 6))
for c in range(K):
    masque = cluster.to_numpy() == c
    ax.scatter(plan[masque, 0], plan[masque, 1], s=12, alpha=0.6, label=f'Groupe {c}')
ax.scatter(centres_plan[:, 0], centres_plan[:, 1], marker='X', s=150, color='black', label='Centres')
ax.set(xlabel=f'PC1 ({100 * acp.explained_variance_ratio_[0]:.1f} %)',
       ylabel=f'PC2 ({100 * acp.explained_variance_ratio_[1]:.1f} %)',
       title=f'K-means (k={K}) projeté sur le plan de l’ACP')
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(OUT / 'projection_clusters.png', dpi=150); plt.show()

## 5 Décrire les groupes

Nous décrivons chaque groupe avec :
- ses **centres standardisés** (écart à la moyenne en nombre d'écarts-types) ;
- les **médianes des valeurs réellement mesurées** dans les unités d'origine (sans les valeurs imputées) ;
- la part de valeurs **manquantes**, pour vérifier que les groupes ne sont pas seulement des motifs d'absence ;
- la répartition des **catégories** de procédé, qui n'ont pas servi au calcul.

In [ ]:
centres = pd.DataFrame(kmeans.cluster_centers_, columns=numeriques,
                       index=[f'Groupe {c}' for c in range(K)])
fig, ax = plt.subplots(figsize=(12, 0.6 * K + 2))
image = ax.imshow(centres.clip(-3, 3), cmap='RdBu_r', vmin=-3, vmax=3, aspect='auto')
ax.set_xticks(range(len(numeriques)), numeriques, rotation=60, ha='right')
ax.set_yticks(range(K), centres.index)
for i in range(K):
    for j in range(len(numeriques)):
        ax.text(j, i, f'{centres.iloc[i, j]:.1f}', ha='center', va='center', fontsize=7)
fig.colorbar(image, ax=ax, label='Écart à la moyenne (écarts-types)')
ax.set_title('Centres des groupes (variables standardisées, bornées à ±3)')
fig.tight_layout(); fig.savefig(OUT / 'centres_clusters.png', dpi=150); plt.show()

In [ ]:
profil_mesure = X[numeriques].groupby(cluster).median().T
profil_mesure.columns = [f'Groupe {c}' for c in profil_mesure.columns]
profil_manquants = (100 * X[numeriques].isna().groupby(cluster).mean()).T
profil_manquants.columns = profil_mesure.columns
print('Médianes des valeurs mesurées :')
display(profil_mesure.round(3))
print('Part de valeurs manquantes (%) :')
display(profil_manquants.round(0))

In [ ]:
for colonne in categories:
    tableau = pd.crosstab(X[colonne].fillna('manquant'), cluster, normalize='columns').mul(100)
    tableau.columns = [f'Groupe {c}' for c in tableau.columns]
    print(f'{colonne} (% des lignes de chaque groupe) :')
    display(tableau.round(0))

### Les groupes de dépôts de Jules sont-ils coupés ?

Les groupes de Jules réunissent les lignes de même composition et mêmes paramètres, avec des traitements thermiques différents. Si un groupe de Jules est réparti dans plusieurs clusters, c'est que le traitement thermique (`PWHT_T_C`, `PWHT_time_h`) pèse dans la distance.

In [ ]:
clusters_par_groupe = cluster.groupby(G).nunique()
multi = clusters_par_groupe[G.value_counts().loc[clusters_par_groupe.index] > 1]
print(f'{(multi > 1).sum()} groupes de Jules sur {len(multi)} (ceux d’au moins 2 lignes) '
      f'sont répartis dans plusieurs clusters.')

## 6 Relier les groupes au score, après coup

Pour les lignes de développement qui ont un score, nous comparons la distribution du score, d'UTS et de l'allongement par groupe. Le score n'a joué aucun rôle dans la construction des groupes.

Les moyennes sont pondérées par groupe de dépôts, comme dans les notebooks 02 à 06.

In [ ]:
dev_score = traction.loc[etiquetees]
assert dev_score['partition'].eq('developpement').all()
lien = dev_score[['UTS_MPa', 'Elongation_pct', 'score_compromis', 'groupe']].join(cluster)
lien['poids'] = 1 / lien['groupe'].map(lien['groupe'].value_counts())

def moyenne_ponderee(t, colonne):
    return np.average(t[colonne], weights=t['poids'])

resume = pd.DataFrame({
    'lignes_avec_score': lien.groupby('cluster').size(),
    'lignes_sans_score': cluster.drop(etiquetees).value_counts(),
    **{f'moyenne_{c}': lien.groupby('cluster').apply(moyenne_ponderee, c, include_groups=False)
       for c in ['score_compromis', 'UTS_MPa', 'Elongation_pct']}
}).fillna(0)
resume.index = [f'Groupe {c}' for c in resume.index]
display(resume.round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
presents = sorted(lien['cluster'].unique())
for ax, colonne, titre in zip(axes, ['score_compromis', 'UTS_MPa', 'Elongation_pct'],
                              ['Score de compromis', 'UTS (MPa)', 'Allongement (%)']):
    ax.boxplot([lien.loc[lien['cluster'] == c, colonne] for c in presents],
               tick_labels=[f'G{c}' for c in presents])
    ax.set(title=titre, xlabel='Groupe K-means')
fig.tight_layout(); fig.savefig(OUT / 'score_par_cluster.png', dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
for c in presents:
    sel = lien[lien['cluster'] == c]
    ax.scatter(sel['UTS_MPa'], sel['Elongation_pct'], s=14, alpha=0.7, label=f'Groupe {c}')
ax.set(xlabel='UTS (MPa)', ylabel='Allongement (%)', title='Mesures de traction colorées par groupe K-means')
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(OUT / 'traction_par_cluster.png', dpi=150); plt.show()

## 7 Le groupe K-means suffit-il à prédire le score ?

Test simple : dans chaque pli de validation du notebook 05, nous ajustons la préparation et K-means sur l'apprentissage seulement, puis nous prédisons pour chaque ligne de validation **le score moyen de son groupe** dans l'apprentissage. Si ce prédicteur fait nettement mieux que la moyenne constante, les groupes trouvés sans la cible portent une partie de l'information utile.

Les lignes sans score ne sont pas utilisées ici, pour rester comparable aux notebooks 05 et 06.

In [ ]:
mesures = ['UTS_MPa', 'Elongation_pct']

def poids_groupes(g):
    return (1 / g.map(g.value_counts())).to_numpy(dtype=float)

def calculer_cible(Y_apprentissage, groupes_apprentissage, Y_a_transformer):
    poids = poids_groupes(groupes_apprentissage)
    rangs = pd.DataFrame(index=Y_a_transformer.index)
    for mesure in mesures:
        tableau = pd.DataFrame({'valeur': Y_apprentissage[mesure].to_numpy(), 'poids': poids})
        masse = tableau.groupby('valeur')['poids'].sum()
        reference = (masse.cumsum() - masse / 2) / masse.sum()
        rangs[mesure] = np.interp(Y_a_transformer[mesure], masse.index, reference)
    return np.sqrt(rangs.prod(axis=1))

def rmse_groupes(y, prediction, g):
    return np.sqrt(mean_squared_error(y, prediction, sample_weight=poids_groupes(g)))

dev = traction['partition'].eq('developpement')
X_dev = traction.loc[dev, variables_X]
Y_dev = traction.loc[dev, mesures]
G_dev = traction.loc[dev, 'groupe']

lignes = []
for pli, (train, val) in enumerate(GroupKFold(n_splits=5).split(X_dev, groups=G_dev), start=1):
    ids_train, ids_val = X_dev.index[train], X_dev.index[val]
    y_train = calculer_cible(Y_dev.loc[ids_train], G_dev.loc[ids_train], Y_dev.loc[ids_train])
    y_val = calculer_cible(Y_dev.loc[ids_train], G_dev.loc[ids_train], Y_dev.loc[ids_val])
    w_train = poids_groupes(G_dev.loc[ids_train])
    taux_pli = X_dev.loc[ids_train].notna().mean()
    num_pli = [c for c in taux_pli[taux_pli >= 0.25].index if c not in categories]
    prep = Pipeline([('mediane', SimpleImputer(strategy='median')),
                     ('standardisation', StandardScaler())])
    Z_train = prep.fit_transform(X_dev.loc[ids_train, num_pli])
    Z_val = prep.transform(X_dev.loc[ids_val, num_pli])
    constante = np.average(y_train, weights=w_train)
    lignes.append({'pli': pli, 'k': 1,
                   'RMSE_groupes': rmse_groupes(y_val, np.full(len(val), constante), G_dev.loc[ids_val])})
    for k in [2, 3, 4, 5, 6, 8, 10, 15, 20, 30]:
        km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Z_train)
        etiquettes_train = km.labels_
        moyennes = pd.Series({c: np.average(y_train[etiquettes_train == c], weights=w_train[etiquettes_train == c])
                              for c in np.unique(etiquettes_train)})
        prediction = moyennes.reindex(km.predict(Z_val)).fillna(constante).to_numpy()
        lignes.append({'pli': pli, 'k': k,
                       'RMSE_groupes': rmse_groupes(y_val, prediction, G_dev.loc[ids_val])})

prediction_cluster = pd.DataFrame(lignes).groupby('k')['RMSE_groupes'].agg(['mean', 'std'])
prediction_cluster.index.name = 'k (1 = moyenne constante)'
display(prediction_cluster.round(4))

comparaison_06 = pd.read_csv(ROOT / 'outputs/06/comparaison_validation.csv').set_index('modele')
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(prediction_cluster.index, prediction_cluster['mean'], yerr=prediction_cluster['std'],
            marker='o', capsize=3, label='Moyenne du groupe K-means')
for nom, style in [('Moyenne constante', ':'), ('Ridge', '--'), ('Forêt aléatoire', '-.')]:
    ax.axhline(comparaison_06.loc[nom, 'RMSE_groupes'], color='grey', linestyle=style, label=nom)
ax.set(xscale='log', xlabel='Nombre de groupes K-means', ylabel='RMSE de validation par groupes',
       title='Prédire le score par la moyenne de son groupe')
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(OUT / 'prediction_par_cluster.png', dpi=150); plt.show()

## 8 Enregistrer les résultats

In [ ]:
criteres.to_csv(OUT / 'criteres_choix_k.csv')
pd.concat([cluster, G.rename('groupe')], axis=1).to_csv(OUT / 'clusters_hors_test.csv', index_label='row_id')
centres.to_csv(OUT / 'centres_standardises.csv')
profil_mesure.to_csv(OUT / 'profil_medianes.csv', index_label='variable')
profil_manquants.to_csv(OUT / 'profil_manquants.csv', index_label='variable')
resume.to_csv(OUT / 'resume_clusters.csv', index_label='cluster')
prediction_cluster.to_csv(OUT / 'prediction_par_cluster.csv')
bilan = {'k_retenu': K, 'variables': numeriques, 'lignes_hors_test': len(X),
         'lignes_avec_score': len(etiquetees), 'seed': SEED, 'sklearn': sklearn_version,
         'regle_choix_k': 'meilleure silhouette parmi k>2 de stabilité bootstrap ARI >= 0.75'}
(OUT / 'bilan.json').write_text(json.dumps(bilan, ensure_ascii=False, indent=2))
print('Résultats enregistrés dans', OUT)

## Conclusion

*À rédiger : nombre de groupes retenu et pourquoi, ce qui distingue les groupes (sections 4 et 5), lien avec le score (section 6) et intérêt pour la prédiction (section 7).*